<a href="https://colab.research.google.com/github/Dani2003/paper-implementations/blob/main/Implementation_of_VisAnomReasoner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# [CELL 1 - Environment Setup]
!pip install -q torch torchvision transformers peft accelerate pillow matplotlib scipy scikit-learn

import os
import io
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.ensemble import IsolationForest

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")

Executing on device: cuda


In [4]:
# [CELL 2 - Plot Renderer & Synthetic Benchmark Generator]

def generate_visanom_sample(seq_len=200, img_size=(400, 200)):
    """
    Generates a time-series signal, injects a localized anomaly,
    renders it as a PIL Image plot, and produces structured rationale annotations.
    """
    t = np.arange(seq_len)
    # Base signal: sine wave + random Gaussian noise
    base_signal = np.sin(2 * np.pi * t / 30.0) + np.random.normal(0, 0.15, size=seq_len)

    # Randomly select an anomaly interval [t_start, t_end]
    anomaly_len = np.random.randint(15, 25)
    t_start = np.random.randint(30, seq_len - anomaly_len - 30)
    t_end = t_start + anomaly_len

    anomaly_type = random.choice(["inverted_spike", "variance_shift", "level_shift"])
    signal = base_signal.copy()

    if anomaly_type == "inverted_spike":
        signal[t_start:t_end] = -3.0 * np.sin(2 * np.pi * t[t_start:t_end] / 30.0)
        desc = f"inverted spike where expected peaks flip sharply to negative troughs."
    elif anomaly_type == "variance_shift":
        signal[t_start:t_end] += np.random.normal(0, 1.2, size=anomaly_len)
        desc = f"sudden high-frequency variance shift with high noise amplitude."
    else: # level_shift
        signal[t_start:t_end] += 2.5
        desc = f"sustained positive baseline shift off the mean."

    # Render time-series to image plot
    fig, ax = plt.subplots(figsize=(img_size[0]/100, img_size[1]/100), dpi=100)
    ax.plot(t, signal, color='#1f77b4', linewidth=1.5)
    ax.set_xlim(0, seq_len)
    ax.set_ylim(-4.5, 4.5)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_xlabel("Time Index (t)", fontsize=8)
    ax.set_ylabel("Value", fontsize=8)
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100)
    plt.close(fig)
    buf.seek(0)
    img = Image.open(buf).convert('RGB')

    # Formulate step-by-step natural language reasoning chain
    rationale = (
        f"Step 1: The plot displays a sequential signal across {seq_len} time units.\n"
        f"Step 2: A noticeable structural deviation occurs between t={t_start} and t={t_end}, showing an {desc}\n"
        f"Step 3: Anomaly located at interval [{t_start}, {t_end}]."
    )

    norm_interval = [t_start / seq_len, t_end / seq_len]
    return img, signal, (t_start, t_end), norm_interval, rationale

# Generate a verification sample
sample_img, sample_sig, raw_bounds, norm_bounds, sample_rationale = generate_visanom_sample()
print(f"Generated Sample Anomaly Bounds: {raw_bounds} (Normalized: {norm_bounds})")
print("Rationale Ground Truth:\n" + sample_rationale)

Generated Sample Anomaly Bounds: (119, 135) (Normalized: [0.595, 0.675])
Rationale Ground Truth:
Step 1: The plot displays a sequential signal across 200 time units.
Step 2: A noticeable structural deviation occurs between t=119 and t=135, showing an sustained positive baseline shift off the mean.
Step 3: Anomaly located at interval [119, 135].


In [5]:
# [CELL 3 - VisAnomBench Dataset Loader]

class VisAnomDataset(Dataset):
    def __init__(self, num_samples=300, seq_len=200):
        self.samples = [generate_visanom_sample(seq_len=seq_len) for _ in range(num_samples)]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img, signal, raw_bounds, norm_bounds, rationale = self.samples[idx]

        # Convert image to RGB Tensor
        img_arr = np.array(img).transpose(2, 0, 1) / 255.0
        img_tensor = torch.tensor(img_arr, dtype=torch.float32)
        signal_tensor = torch.tensor(signal, dtype=torch.float32)
        bounds_tensor = torch.tensor(norm_bounds, dtype=torch.float32)

        return {
            "image": img_tensor,
            "signal": signal_tensor,
            "bounds": bounds_tensor,
            "raw_bounds": raw_bounds,
            "rationale": rationale
        }

train_dataset = VisAnomDataset(num_samples=400)
test_dataset  = VisAnomDataset(num_samples=100)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Dataset Initialized: Train={len(train_dataset)} | Test={len(test_dataset)}")

Dataset Initialized: Train=400 | Test=100


In [6]:
# [CELL 4 - VisAnomReasoner Model Definition]

class VisualEncoder(nn.Module):
    """Extracts spatial visual features from time-series line plot images."""
    def __init__(self, embed_dim=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1), # [32, 100, 200]
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), # [64, 50, 100]
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), # [128, 25, 50]
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 16)) # Bottleneck feature grid
        )
        self.proj = nn.Linear(128 * 16, embed_dim)

    def forward(self, x):
        feat = self.conv(x)
        feat = feat.view(feat.size(0), -1)
        return self.proj(feat)

class VisAnomReasoner(nn.Module):
    """
    Vision-Language Reasoner for Time-Series Anomaly Detection.
    Predicts normalized anomaly interval bounds [t_start, t_end] and latent reasoning vectors.
    """
    def __init__(self, embed_dim=256):
        super().__init__()
        self.visual_encoder = VisualEncoder(embed_dim=embed_dim)

        # Multimodal Transformer Fusion Core
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=8, dim_feedforward=512, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)

        # Continuous Interval Bounding Head [t_start, t_end]
        self.interval_head = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 2),
            nn.Sigmoid() # Bounds constrained to [0, 1]
        )

    def forward(self, images):
        img_feats = self.visual_encoder(images).unsqueeze(1) # [B, 1, embed_dim]
        fused_feats = self.transformer(img_feats).squeeze(1) # [B, embed_dim]
        pred_bounds = self.interval_head(fused_feats)
        return pred_bounds

model = VisAnomReasoner().to(device)
print("VisAnomReasoner Architecture Loaded Successfully.")

VisAnomReasoner Architecture Loaded Successfully.


In [7]:
# [CELL 5 - Multi-Task Training Loop]

def calculate_interval_iou(pred_bounds, true_bounds):
    """Calculates 1D Intersection over Union (IoU) between predicted and true intervals."""
    pred_start, pred_end = torch.min(pred_bounds, dim=-1)[0], torch.max(pred_bounds, dim=-1)[0]
    true_start, true_end = true_bounds[:, 0], true_bounds[:, 1]

    intersection_start = torch.maximum(pred_start, true_start)
    intersection_end   = torch.minimum(pred_end, true_end)
    intersection = torch.clamp(intersection_end - intersection_start, min=0.0)

    union = (pred_end - pred_start) + (true_end - true_start) - intersection
    iou = intersection / (union + 1e-6)
    return iou.mean()

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.SmoothL1Loss()

epochs = 25
print("Starting VisAnomReasoner Fine-Tuning on VisAnomBench...")

for epoch in range(1, epochs + 1):
    model.train()
    total_loss, total_iou = 0.0, 0.0

    for batch in train_loader:
        images = batch["image"].to(device)
        gt_bounds = batch["bounds"].to(device)

        optimizer.zero_grad()
        pred_bounds = model(images)

        # Interval Boundary Regression Loss
        loss = criterion(pred_bounds, gt_bounds)
        loss.backward()
        optimizer.step()

        iou = calculate_interval_iou(pred_bounds, gt_bounds)
        total_loss += loss.item()
        total_iou += iou.item()

    if epoch % 5 == 0 or epoch == 1:
        avg_loss = total_loss / len(train_loader)
        avg_iou = total_iou / len(train_loader)
        print(f"Epoch [{epoch:02d}/{epochs}] | Loss: {avg_loss:.4f} | Interval IoU: {avg_iou:.4f}")

Starting VisAnomReasoner Fine-Tuning on VisAnomBench...
Epoch [01/25] | Loss: 0.0176 | Interval IoU: 0.1068
Epoch [05/25] | Loss: 0.0161 | Interval IoU: 0.1093
Epoch [10/25] | Loss: 0.0163 | Interval IoU: 0.1094
Epoch [15/25] | Loss: 0.0161 | Interval IoU: 0.1091
Epoch [20/25] | Loss: 0.0161 | Interval IoU: 0.1086
Epoch [25/25] | Loss: 0.0160 | Interval IoU: 0.1102


In [8]:
# [CELL 6 - Evaluation Benchmark & Anomaly Grounding Comparison]

def evaluate_visanom():
    model.eval()
    all_preds, all_gts = [], []

    # 1. Evaluate VisAnomReasoner
    with torch.no_grad():
        for batch in test_loader:
            images = batch["image"].to(device)
            pred_bounds = model(images).cpu().numpy()
            gt_bounds = batch["bounds"].numpy()
            all_preds.append(pred_bounds)
            all_gts.append(gt_bounds)

    all_preds = np.vstack(all_preds)
    all_gts   = np.vstack(all_gts)

    # Convert back to index scale (seq_len=200)
    pred_indices = (all_preds * 200).astype(int)
    gt_indices   = (all_gts * 200).astype(int)

    # 2. Benchmark against classical Isolation Forest baseline
    iforest_ious = []
    reasoner_ious = []

    for i in range(len(test_dataset)):
        sample = test_dataset[i]
        sig = sample["signal"].numpy().reshape(-1, 1)
        gt_s, gt_e = sample["raw_bounds"]

        # Isolation Forest Detection
        clf = IsolationForest(contamination=0.1, random_state=42)
        preds = clf.fit_predict(sig)
        anom_idxs = np.where(preds == -1)[0]

        if len(anom_idxs) > 0:
            if_s, if_e = anom_idxs.min(), anom_idxs.max()
        else:
            if_s, if_e = 0, 0

        # Calculate IoU for IsolationForest
        inter = max(0, min(if_e, gt_e) - max(if_s, gt_s))
        union = max(1, max(if_e, gt_e) - min(if_s, gt_s))
        iforest_ious.append(inter / union)

        # Calculate IoU for VisAnomReasoner
        pr_s, pr_e = pred_indices[i]
        r_inter = max(0, min(pr_e, gt_e) - max(pr_s, gt_s))
        r_union = max(1, max(pr_e, gt_e) - min(pr_s, gt_s))
        reasoner_ious.append(r_inter / r_union)

    print("\n" + "=" * 60)
    print("BENCHMARK EVALUATION RESULTS (VisAnomBench Test Set)")
    print("=" * 60)
    print(f"Isolation Forest Baseline Mean Interval IoU : {np.mean(iforest_ious)*100:.2f}%")
    print(f"VisAnomReasoner (Ours)    Mean Interval IoU : {np.mean(reasoner_ious)*100:.2f}%")
    print("=" * 60)
    print(f"Net Improvement over Baseline              : +{(np.mean(reasoner_ious) - np.mean(iforest_ious))*100:.2f} pp")

evaluate_visanom()


BENCHMARK EVALUATION RESULTS (VisAnomBench Test Set)
Isolation Forest Baseline Mean Interval IoU : 15.54%
VisAnomReasoner (Ours)    Mean Interval IoU : 9.84%
Net Improvement over Baseline              : +-5.70 pp
